# EXTRA | Connecting Python and MySQL

## Learning objectives

By the end of this notebook, you should be able to:

- connect Python to a local MySQL database;
- execute SQL queries from Python;
- load query results directly into a Pandas DataFrame;
- use query parameters instead of building SQL with string interpolation;
- create Python visualizations from data stored in MySQL;
- understand how this workflow automates the manual bridge between SQL and Python.

---

## Why connect Python and MySQL?

So far, you have mainly worked with the two tools separately:

```text
MySQL Workbench
      ↓
run SQL query
      ↓
inspect / export result
      ↓
Python
      ↓
Pandas + visualization
```

A direct connection lets Python query MySQL itself:

```text
Python
   ↓
SQLAlchemy
   ↓
MySQL
   ↓
SQL query
   ↓
Pandas DataFrame
   ↓
visualization
```

This does **not** give SQL or Python new analytical capabilities.

It simply makes the workflow easier to automate and reproduce.

> This lesson is **EXTRA / SELF-LEARNING**. A direct Python–MySQL connection is not required to complete the SQL Quest.

## 1. Install the required libraries

This notebook uses:

- `SQLAlchemy` — manages the database connection and SQL execution;
- `PyMySQL` — MySQL driver used by SQLAlchemy;
- `pandas` — stores query results in DataFrames;
- `matplotlib` — creates visualizations.

If your project uses `uv`, run this command from the project folder:

```bash
uv add sqlalchemy pymysql pandas matplotlib
```

Then make sure your Jupyter kernel is using the project's virtual environment.

In [ ]:
import getpass

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import URL, create_engine, text

## 2. Define the connection settings

We will connect to the same local `bank` database used in the SQL lessons.

Do **not** hard-code your password inside the notebook.

`getpass.getpass()` asks for it without displaying it on screen.

In [ ]:
host = "localhost"
port = 3306
database = "bank"
username = "root"

password = getpass.getpass("MySQL password: ")

## 3. Create a SQLAlchemy Engine

The **Engine** contains the information SQLAlchemy needs to connect to the database.

Using `URL.create()` is preferable to manually concatenating a connection string because it safely handles special characters that may appear in passwords.

In [ ]:
url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database,
)

engine = create_engine(url)

At this point, the Engine knows **how** to connect.

We still need to open a connection when we want to communicate with MySQL.

## 4. Test the connection

Use a context manager:

```python
with engine.connect() as connection:
    ...
```

The connection is released automatically when the `with` block finishes.

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1 AS connection_test;"))
    print(result.one())

Expected result:

```text
(1,)
```

If this fails, check:

- that MySQL is running;
- the username and password;
- the host and port;
- that the `bank` database exists.

Do not continue until the connection works.

## 5. Execute a SQL query from Python

SQLAlchemy's `text()` function lets us define a textual SQL statement.

Start with a small query and validate the result before building anything more complicated.

In [ ]:
query = text("""
SELECT loan_id,
       account_id,
       amount,
       duration,
       status
FROM loan
LIMIT 10;
""")

with engine.connect() as connection:
    result = connection.execute(query)
    rows = result.all()

rows

The database has returned rows to Python.

For Data Analytics, however, we usually want the result in a Pandas DataFrame rather than spending time working directly with SQLAlchemy `Result` objects.

## 6. Load the result directly into Pandas

`pd.read_sql()` can execute the query and return the result as a DataFrame.

In [ ]:
with engine.connect() as connection:
    loans_df = pd.read_sql(query, connection)

loans_df

Now the workflow has crossed the SQL/Python boundary:

```text
MySQL table
    ↓
SQL query
    ↓
SQLAlchemy connection
    ↓
Pandas DataFrame
```

From this point onward, `loans_df` is an ordinary Pandas DataFrame.

In [ ]:
loans_df.info()

## 7. Query only the data you actually need

Avoid using Python to retrieve an entire database table and then reproduce SQL operations unnecessarily.

If filtering or aggregation belongs naturally in SQL, do it in SQL.

For example, suppose we want the number of loans and average loan amount for each status:

In [ ]:
query = text("""
SELECT status,
       COUNT(*) AS loan_count,
       ROUND(AVG(amount), 2) AS average_amount
FROM loan
GROUP BY status
ORDER BY loan_count DESC;
""")

with engine.connect() as connection:
    loan_summary_df = pd.read_sql(query, connection)

loan_summary_df

This keeps the responsibilities clear:

```text
SQL
→ retrieve / filter / join / aggregate

Python
→ further analysis / visualization / communication
```

## 8. Use query parameters

Sometimes a Python value needs to be passed into a SQL query.

Avoid constructing SQL with f-strings such as:

```python
# Avoid this pattern
f"SELECT * FROM loan WHERE amount > {minimum_amount}"
```

Use SQL parameters instead.

In [ ]:
minimum_amount = 100_000

query = text("""
SELECT loan_id,
       account_id,
       amount,
       duration,
       status
FROM loan
WHERE amount > :minimum_amount
ORDER BY amount DESC;
""")

with engine.connect() as connection:
    large_loans_df = pd.read_sql(
        query,
        connection,
        params={"minimum_amount": minimum_amount},
    )

large_loans_df.head()

The SQL statement contains:

```sql
:minimum_amount
```

and Python supplies its value separately.

This makes the query clearer and avoids manually injecting values into SQL strings.

## 9. Build SQL incrementally — exactly as before

Connecting Python to MySQL does **not** change the way you should solve SQL problems.

Do not write a complex SQL query inside Python in one attempt.

Use the same workflow:

```text
write SQL in small pieces
        ↓
run
        ↓
inspect result
        ↓
validate
        ↓
add next piece
        ↓
validate again
```

If a query becomes complex, it is often easier to prototype and debug it in MySQL Workbench first, then move the validated query into Python.

## 10. Visualize the SQL result with Python

This is the main reason this workflow is useful for the SQL Quest.

We can query MySQL and immediately visualize the result in Python.

Let's use the summary DataFrame created earlier.

In [ ]:
loan_summary_df

In [ ]:
ax = loan_summary_df.plot(
    x="status",
    y="loan_count",
    kind="bar",
    legend=False,
)

ax.set_title("Number of loans by status")
ax.set_xlabel("Loan status")
ax.set_ylabel("Number of loans")

plt.tight_layout()
plt.show()

The complete workflow is now:

```text
BUSINESS QUESTION
        ↓
SQL QUERY
        ↓
MYSQL
        ↓
PANDAS DATAFRAME
        ↓
PYTHON VISUALIZATION
        ↓
INTERPRETATION
```

This is the same workflow required by the Quest, but the transfer from MySQL to Python is now automated.

## 11. A reusable helper function

If you need to run several read-only queries, a small helper can reduce repetition.

The function below receives:

- a SQLAlchemy Engine;
- a SQL query;
- optional query parameters;

and returns a DataFrame.

In [ ]:
def query_to_dataframe(engine, query, params=None):
    """Execute a read-only SQL query and return the result as a DataFrame."""

    with engine.connect() as connection:
        return pd.read_sql(
            text(query),
            connection,
            params=params,
        )

In [ ]:
sql = """
SELECT account_id,
       COUNT(*) AS transaction_count
FROM trans
GROUP BY account_id
ORDER BY transaction_count DESC
LIMIT 10;
"""

top_accounts_df = query_to_dataframe(engine, sql)
top_accounts_df

This function is deliberately small.

Do not hide complex SQL logic inside a Python function before validating the SQL itself.

First:

```text
SQL query
   ↓
validate
```

Then automate the repeated connection/query workflow.

## 12. Connection and credential good practices

For this introductory workflow:

- never commit passwords to GitHub;
- do not write passwords directly in notebooks;
- use `getpass` for interactive practice;
- use query parameters instead of interpolating Python values into SQL strings;
- use `with engine.connect()` so connections are released automatically;
- retrieve only the data required for the analysis.

For larger applications, credentials are normally managed with environment variables or dedicated secret-management systems rather than entered interactively.

## 13. What about ORM and `mysql-connector-python`?

SQLAlchemy also provides an **Object-Relational Mapper (ORM)** that maps database structures to Python classes.

ORMs are important in software development, but they are outside the scope of this Data Analytics bootcamp.

MySQL also provides other Python connectors, including `mysql-connector-python`.

For this lesson, we deliberately use **one workflow only**:

```text
SQLAlchemy
   +
PyMySQL
   +
Pandas
```

Learning multiple connection libraries would not improve the analytical objective of this lesson.

## Check for Understanding

Try the following using the `bank` database.

### 1. Basic query

Retrieve the first 20 accounts into a DataFrame.

Include:

- `account_id`
- `district_id`
- `frequency`
- `date`

Validate the resulting DataFrame.

### 2. SQL aggregation → DataFrame

Use SQL to calculate the number of accounts for each `frequency`.

Load the result into Pandas.

### 3. Visualize

Create a Python visualization of the result from exercise 2.

### 4. Parameterized query

Create a Python variable:

```python
minimum_amount = 200_000
```

Retrieve loans above that amount using a SQL parameter.

Do **not** insert the value using an f-string.

### 5. Connect with your SQL Quest

Choose one validated SQL query from your Quest.

Use Python to:

```text
connect to MySQL
      ↓
execute the query
      ↓
load the result into a DataFrame
      ↓
create one visualization
```

## Key Takeaway

Connecting Python and MySQL does not replace SQL or Pandas.

It connects them.

```text
SQL
→ retrieve and prepare data


Python
→ analyze and visualize


SQLAlchemy
→ bridge between them
```

The important analytical workflow remains:

```text
QUESTION
   ↓
SQL
   ↓
VALIDATE
   ↓
DATAFRAME
   ↓
VALIDATE
   ↓
VISUALIZE
   ↓
INTERPRET
```

This lesson is optional, but it shows how the SQL and Python skills you already know can become part of one reproducible workflow.